In [2]:
import os
import re
from pathlib import Path

import pandas as pd
import tkinter as tk
from tkinter import ttk
from PIL import Image, ImageTk

# Mengubah direktori kerja ke folder proyek agar path relatif berfungsi
os.chdir(r"C:\Users\Pongo\Sistem_Rekomendasi_Wisata")

DATASET_PATH = Path("Dataset Wisata Magelang/Dataset_tourisMagelang.csv")
GAMBAR_DIR   = Path("Image Wisata")

# Validasi keberadaan file dan folder sebelum program berjalan
assert DATASET_PATH.exists(), f"Dataset tidak ditemukan: {DATASET_PATH.resolve()}"
assert GAMBAR_DIR.exists(),   f"Folder gambar tidak ditemukan: {GAMBAR_DIR.resolve()}"

df = pd.read_csv(DATASET_PATH)

# Palet Warna
BG_COLOR     = "#EFEAE1"
PRIMARY      = "#1B2A41"
SECONDARY    = "#C9A227"
SECONDARY_HO = "#B08D1F"
TEXT_DARK    = "#1B2A41"
TEXT_MUTED   = "#6B7A8F"
TEXT_LIGHT   = "#F5F1E8"
ACCENT       = "#7A8B99"
CARD_BG      = "#FFFFFF"
CARD_BORDER  = "#E6E1D6"
CARD_HOVER   = "#FBF9F4"
SHADOW       = "#DCD6C9"
IMAGE_BG     = "#E8E3D7"


def ambil_kota(alamat):
    """Mengekstrak nama kota/kabupaten dari string alamat lengkap.
    Jika tidak ditemukan, ambil elemen kedua dari belakang."""
    if not isinstance(alamat, str):
        return "-"
    m = re.search(r"(Kota Magelang|Kabupaten Magelang)", alamat)
    if m:
        return m.group(1)
    bagian = [b.strip() for b in alamat.split(",") if b.strip()]
    return bagian[-2] if len(bagian) >= 2 else "-"


def format_rupiah(harga):
    """Mengubah angka (float/int) menjadi format Rupiah (Rp).
    Jika 0 atau NaN, akan mengembalikan string 'Gratis'."""
    try:
        h = float(harga)
        if pd.isna(h) or h == 0:
            return "Gratis"
        return "Rp" + f"{int(h):,}".replace(",", ".")
    except (TypeError, ValueError):
        return "-"


_cache_gambar = {}

def cari_gambar(nama):
    """Mencari file gambar di folder GAMBAR_DIR berdasarkan nama destinasi.
    Mengembalikan objek Path jika ketemu, atau None jika tidak ada."""
    for ext in (".jpg", ".jpeg", ".png", ".webp", ".gif", ".bmp"):
        p = GAMBAR_DIR / f"{nama}{ext}"
        if p.exists():
            return p
    # Fallback case-insensitive jika ekstensi persis tidak ketemu
    nama_low = nama.lower()
    for p in GAMBAR_DIR.iterdir():
        if p.is_file() and p.stem.lower() == nama_low:
            return p
    return None


def load_thumb(nama, ukuran=(380, 180)):
    """Memuat gambar dari disk, melakukan crop tengah (center-crop), 
    dan mengubah ukurannya sesuai parameter 'ukuran'.
    Hasilnya disimpan di cache agar tidak memuat ulang (hemat memori)."""
    key = (nama, ukuran)
    if key in _cache_gambar:
        return _cache_gambar[key]

    path = cari_gambar(nama)
    if path is None:
        _cache_gambar[key] = None
        return None

    try:
        img = Image.open(path).convert("RGB")
        tw, th = ukuran
        w, h = img.size
        # Crop tengah agar rasio sesuai target tanpa merusak proporsi
        if w / h > tw / th:
            nw = int(h * tw / th)
            left = (w - nw) // 2
            img = img.crop((left, 0, left + nw, h))
        else:
            nh = int(w * th / tw)
            top = (h - nh) // 2
            img = img.crop((0, top, w, top + nh))
        img = img.resize(ukuran, Image.LANCZOS)
        photo = ImageTk.PhotoImage(img)
        _cache_gambar[key] = photo
        return photo
    except Exception as e:
        print(f"⚠ Gagal memuat {path}: {e}")
        _cache_gambar[key] = None
        return None


def bangun_data_wisata(frame):
    #Membaca DataFrame dari CSV dan mengonversinya menjadi list of dict.
    hasil = []
    for _, row in frame.iterrows():
        hw = row.get("Weekday_Price", 0)
        he = row.get("Weekend_Holiday_Price", 0)
        harga_acuan = hw if (hw and not pd.isna(hw)) else he
        try:
            rating = float(row.get("Rating_Gmaps", 0))
        except (TypeError, ValueError):
            rating = 0.0

        hasil.append({
            "id":             str(row.get("Place_Id", "-")),
            "nama":           str(row.get("Place_Name", "")).strip(),
            "lokasi":         ambil_kota(row.get("Address", "")),
            "kategori":       str(row.get("Category", "")).strip(),
            "deskripsi":      str(row.get("Description", "")).strip(),
            "harga":          format_rupiah(harga_acuan),
            "harga_weekday":  format_rupiah(hw),
            "harga_weekend":  format_rupiah(he),
            "rating":         rating,
            "alamat":         str(row.get("Address", "")).strip(),
        })
    return hasil


DATA_WISATA = bangun_data_wisata(df)
KATEGORI_UNIK = sorted({d["kategori"] for d in DATA_WISATA if d["kategori"]})


class AplikasiWisata(tk.Tk):
    def __init__(self):
        """Inisialisasi jendela utama, mengatur ukuran, dan memanggil
        fungsi-fungsi untuk membangun antarmuka (header, toolbar, list)."""
        super().__init__()
        self.title("Rekomendasi Wisata Magelang")
        try:
            self.state("zoomed")
        except tk.TclError:
            self.attributes("-zoomed", True)
        self.configure(bg=BG_COLOR)
        self.minsize(980, 640)

        self._placeholder = "Cari destinasi, kategori, atau lokasi..."
        self._placeholder_active = True
        self._entry = None
        self._canvas = None
        self._current_cols = 2

        self._items_tampil = []
        self._card_widgets = []
        self._gambar_refs = []
        self._popup = None

        self._build_header()
        self._build_toolbar()
        self._build_list_area()

        self._apply_filter(force=True)

    def _build_header(self):
        """Membangun bagian atas jendela (judul aplikasi dan counter)."""
        header = tk.Frame(self, bg=PRIMARY, height=110)
        header.pack(fill="x")
        header.pack_propagate(False)

        tk.Frame(self, bg=SECONDARY, height=3).pack(fill="x")

        inner = tk.Frame(header, bg=PRIMARY)
        inner.pack(fill="both", expand=True, padx=40, pady=20)

        left = tk.Frame(inner, bg=PRIMARY)
        left.pack(side="left", fill="y")

        tk.Label(left, text="✈  Rekomendasi Wisata Magelang", bg=PRIMARY, fg=TEXT_LIGHT, font=("Georgia", 22, "bold")).pack(anchor="w")
        tk.Label(left, text="Temukan destinasi impianmu di Magelang", bg=PRIMARY, fg=SECONDARY, font=("Georgia", 11, "italic")).pack(anchor="w", pady=(4, 0))

        right = tk.Frame(inner, bg=PRIMARY)
        right.pack(side="right", fill="y")

        self.header_counter = tk.Label(right, text="0", bg=PRIMARY, fg=TEXT_LIGHT, font=("Georgia", 22, "bold"))
        self.header_counter.pack(anchor="e")
        tk.Label(right, text="destinasi ditemukan", bg=PRIMARY, fg=SECONDARY, font=("Segoe UI", 9)).pack(anchor="e")

    def _build_toolbar(self):
        """Membangun baris pencarian, dropdown filter kategori, dan tombol reset."""
        bar = tk.Frame(self, bg=BG_COLOR)
        bar.pack(fill="x", padx=40, pady=(24, 10))

        search_wrap = tk.Frame(bar, bg=CARD_BG, highlightbackground=CARD_BORDER, highlightthickness=1)
        search_wrap.pack(side="left", fill="x", expand=True)

        tk.Label(search_wrap, text="🔍", bg=CARD_BG, fg=ACCENT, font=("Segoe UI", 13)).pack(side="left", padx=(14, 6), pady=10)

        self.search_var = tk.StringVar()
        self._entry = tk.Entry(search_wrap, textvariable=self.search_var, bg=CARD_BG, fg=TEXT_MUTED, relief="flat", font=("Segoe UI", 11), insertbackground=PRIMARY)
        self._entry.pack(side="left", fill="x", expand=True, padx=(0, 14), pady=10)
        self._entry.focus_set()

        self.search_var.set(self._placeholder)
        self._placeholder_active = True

        self._entry.bind("<FocusIn>", self._on_focus_in)
        self._entry.bind("<FocusOut>", self._on_focus_out)
        self._entry.bind("<KeyRelease>", lambda e: self._apply_filter())

        tk.Frame(bar, bg=BG_COLOR, width=14).pack(side="left")

        filter_wrap = tk.Frame(bar, bg=BG_COLOR)
        filter_wrap.pack(side="left")
        tk.Label(filter_wrap, text="Kategori", bg=BG_COLOR, fg=TEXT_MUTED, font=("Segoe UI", 9, "bold")).pack(anchor="w")

        self.filter_var = tk.StringVar(value="Semua Kategori")
        self.filter_box = ttk.Combobox(filter_wrap, textvariable=self.filter_var, values=["Semua Kategori"] + KATEGORI_UNIK, state="readonly", width=20, font=("Segoe UI", 10))
        self.filter_box.pack(pady=(2, 0))
        self.filter_box.bind("<<ComboboxSelected>>", lambda e: self._apply_filter())

        self.btn_reset = tk.Button(bar, text="Reset", bg=SECONDARY, fg=PRIMARY, font=("Segoe UI", 10, "bold"), activebackground=SECONDARY_HO, activeforeground=PRIMARY, relief="flat", padx=20, pady=7, cursor="hand2", borderwidth=0, command=self._reset)
        self.btn_reset.pack(side="left", padx=(12, 0), pady=(16, 0))

    def _on_focus_in(self, e):
        """Menghapus teks placeholder saat kotak pencarian mulai diklik/difokuskan."""
        if self.search_var.get() == self._placeholder:
            self._placeholder_active = False
            self.search_var.set("")
            self._entry.config(fg=TEXT_DARK)

    def _on_focus_out(self, e):
        """Memunculkan kembali teks placeholder jika kotak pencarian kosong dan ditinggalkan."""
        isi = self.search_var.get().strip()
        if isi == "" or isi == self._placeholder:
            self._placeholder_active = True
            self.search_var.set(self._placeholder)
            self._entry.config(fg=TEXT_MUTED)

    def _build_list_area(self):
        """Membangun area scrollable (Canvas + Frame) untuk menampung card-card wisata."""
        wrapper = tk.Frame(self, bg=BG_COLOR)
        wrapper.pack(fill="both", expand=True, padx=40, pady=(6, 20))

        self._canvas = tk.Canvas(wrapper, bg=BG_COLOR, highlightthickness=0)
        self._scrollbar = ttk.Scrollbar(wrapper, orient="vertical", command=self._canvas.yview)
        self.list_frame = tk.Frame(self._canvas, bg=BG_COLOR)

        self.list_frame.bind("<Configure>", lambda e: self._canvas.configure(scrollregion=self._canvas.bbox("all")))
        self._canvas_window = self._canvas.create_window((0, 0), window=self.list_frame, anchor="nw")
        self._canvas.configure(yscrollcommand=self._scrollbar.set)

        self._canvas.pack(side="left", fill="both", expand=True)
        self._canvas.bind("<Configure>", self._on_canvas_resize)
        # Mendukung scroll pakai mouse wheel
        self._canvas.bind_all("<MouseWheel>", lambda e: self._canvas.yview_scroll(int(-e.delta / 120), "units"))

    def _on_canvas_resize(self, event):
        """Mendeteksi perubahan lebar jendela. Jika lebar melewati batas tertentu,
        jumlah kolom akan berubah (1, 2, atau 3), lalu me-render ulang."""
        self._canvas.itemconfig(self._canvas_window, width=event.width)
        w = event.width
        cols = 3 if w >= 1200 else (2 if w >= 780 else 1)
        if cols != self._current_cols:
            self._current_cols = cols
            self._apply_filter(force=True)

    def _apply_filter(self, force=False):
        """Menyaring DATA_WISATA berdasarkan teks pencarian dan filter kategori.
        Jika datanya sama persis dengan sebelumnya, render dilewati agar tidak berkedip."""
        raw = self.search_var.get().strip()
        q = "" if raw == "" or raw == self._placeholder else raw.lower()
        kategori = self.filter_var.get()

        hasil = DATA_WISATA
        if kategori and kategori != "Semua Kategori":
            hasil = [d for d in hasil if d["kategori"] == kategori]

        if q:
            hasil = [d for d in hasil if q in d["nama"].lower() or q in d["lokasi"].lower() or q in d["kategori"].lower() or q in d["deskripsi"].lower()]

        if not force and hasil == self._items_tampil:
            return

        self._items_tampil = hasil
        self._render(hasil)

    def _render(self, items):
        """Menghapus card lama dan membuat ulang card baru berdasarkan data yang sudah difilter.
        Menghitung jumlah kolom, mengatur lebar baris agar seragam (uniform), dan mengatur wraplength teks."""
        yview = self._canvas.yview()
        for w in self.list_frame.winfo_children():
            w.destroy()
        self._card_widgets.clear()
        self._gambar_refs.clear()

        for r in range(50):
            self.list_frame.grid_rowconfigure(r, weight=0)

        self.header_counter.config(text=str(len(items)))

        if not items:
            self._render_empty()
            return

        cols = self._current_cols
        for c in range(3):
            self.list_frame.grid_columnconfigure(c, weight=0)
        for c in range(cols):
            self.list_frame.grid_columnconfigure(c, weight=1, uniform="card")

        # Hitung lebar untuk wraplength dinamis
        self.update_idletasks()
        lebar_canvas = self._canvas.winfo_width()
        if lebar_canvas < 50:
            lebar_canvas = max(self.winfo_width() - 80, 600)
        lebar_card = max(260, (lebar_canvas - (cols * 20 + 20)) // max(cols, 1))
        wrap_desc = max(180, lebar_card - 50)
        wrap_judul = max(140, lebar_card - 130)

        for idx, data in enumerate(items):
            row, col = divmod(idx, cols)
            info = self._buat_card(data, row, col, wrap_desc, wrap_judul)
            self._card_widgets.append(info)

        # Paksa semua baris tinggi sama (uniform="row")
        n_rows = (len(items) + cols - 1) // cols
        for r in range(n_rows):
            self.list_frame.grid_rowconfigure(r, weight=1, uniform="row")

        self.update_idletasks()
        if yview and yview != (0.0, 1.0):
            self._canvas.yview_moveto(yview[0])

    def _render_empty(self):
        """Menampilkan UI 'Destinasi tidak ditemukan' ketika hasil filter kosong."""
        self._card_widgets.clear()
        box = tk.Frame(self.list_frame, bg=CARD_BG, highlightbackground=CARD_BORDER, highlightthickness=1)
        box.pack(fill="x", pady=20)
        inner = tk.Frame(box, bg=CARD_BG)
        inner.pack(pady=40)
        tk.Label(inner, text="", bg=CARD_BG, fg=ACCENT, font=("Segoe UI", 40)).pack()
        tk.Label(inner, text="Destinasi tidak ditemukan", bg=CARD_BG, fg=PRIMARY, font=("Georgia", 15, "bold")).pack(pady=(8, 4))
        tk.Label(inner, text="Coba kata kunci lain seperti 'candi', 'alam', atau 'Borobudur'", bg=CARD_BG, fg=TEXT_MUTED, font=("Segoe UI", 10)).pack()

    def _buat_card(self, data, row, col, wrap_desc=360, wrap_judul=300):
        """Membuat satu kotak UI (Card) yang berisi gambar, nama, kategori, harga, rating, dan deskripsi singkat.
        Card ini dilengkapi event hover (ubah warna) dan click (buka popup)."""
        nama, lokasi, kategori, deskripsi, harga, rating = data["nama"], data["lokasi"], data["kategori"], data["deskripsi"], data["harga"], data["rating"]

        deskripsi_pendek = deskripsi
        if len(deskripsi_pendek) > 160:
            deskripsi_pendek = deskripsi_pendek[:157].rsplit(" ", 1)[0] + "..."

        outer = tk.Frame(self.list_frame, bg=SHADOW)
        outer.grid(row=row, column=col, sticky="nsew", padx=10, pady=10)
        outer.grid_columnconfigure(0, weight=1)
        outer.grid_rowconfigure(0, weight=1)

        card = tk.Frame(outer, bg=CARD_BG)
        card.grid(row=0, column=0, sticky="nsew", padx=(0, 2), pady=(0, 2))
        card.grid_columnconfigure(0, weight=1)
        card.grid_rowconfigure(1, weight=1)

        target_gambar = []
        thumb = load_thumb(nama, ukuran=(380, 180))
        if thumb is not None:
            self._gambar_refs.append(thumb)
            img_frame = tk.Frame(card, bg=IMAGE_BG)
            img_frame.grid(row=0, column=0, sticky="ew")
            img_frame.grid_propagate(False)
            img_frame.configure(height=180)

            lbl_img = tk.Label(img_frame, image=thumb, bg=IMAGE_BG, bd=0)
            lbl_img.image = thumb
            lbl_img.pack(fill="both", expand=True)
            target_gambar = [img_frame, lbl_img]
        else:
            ph = tk.Frame(card, bg=IMAGE_BG, height=140)
            ph.grid(row=0, column=0, sticky="ew")
            ph.grid_propagate(False)
            lbl_ph = tk.Label(ph, text="gambar belum tersedia", bg=IMAGE_BG, fg=TEXT_MUTED, font=("Segoe UI", 10, "italic"))
            lbl_ph.pack(expand=True)
            target_gambar = [ph, lbl_ph]

        body = tk.Frame(card, bg=CARD_BG)
        body.grid(row=1, column=0, sticky="nsew", padx=20, pady=16)

        top = tk.Frame(body, bg=CARD_BG)
        top.pack(fill="x")

        lbl_nama = tk.Label(top, text=nama, bg=CARD_BG, fg=PRIMARY, font=("Georgia", 14, "bold"), wraplength=wrap_judul, justify="left", anchor="w")
        lbl_nama.pack(side="left", fill="x", expand=True)

        badge = tk.Label(top, text=kategori, bg=SECONDARY, fg=PRIMARY, font=("Segoe UI", 8, "bold"), padx=10, pady=3)
        badge.pack(side="right", anchor="ne")

        meta = tk.Frame(body, bg=CARD_BG)
        meta.pack(fill="x", pady=(8, 0))
        tk.Label(meta, text=f"📍 {lokasi}", bg=CARD_BG, fg=TEXT_MUTED, font=("Segoe UI", 9)).pack(side="left")
        tk.Label(meta, text="•", bg=CARD_BG, fg=CARD_BORDER, font=("Segoe UI", 9)).pack(side="left", padx=8)
        tk.Label(meta, text=f"🎟 {harga}", bg=CARD_BG, fg=TEXT_MUTED, font=("Segoe UI", 9)).pack(side="left")

        lbl_rating = tk.Label(body, text=f"⭐ {rating:.1f} / 5.0", bg=CARD_BG, fg=SECONDARY, font=("Segoe UI", 9, "bold"))
        lbl_rating.pack(anchor="w", pady=(8, 0))

        lbl_desc = tk.Label(body, text=deskripsi_pendek, bg=CARD_BG, fg=TEXT_DARK, font=("Segoe UI", 10), wraplength=wrap_desc, justify="left", anchor="w")
        lbl_desc.pack(fill="x", pady=(8, 0))

        # Menyimpan referensi widget yang harus berubah warna saat hover
        kandidat = [card, body, top, meta, lbl_nama, lbl_rating, lbl_desc] + target_gambar
        for child in meta.winfo_children():
            kandidat.append(child)

        warna_asli = {}
        for w in kandidat:
            try:
                warna_asli[w] = w.cget("bg")
            except tk.TclError:
                pass

        def repaint(warna):
            for widget, asli in warna_asli.items():
                if asli == CARD_BG:
                    try:
                        widget.config(bg=warna)
                    except tk.TclError:
                        pass

        def on_enter(_): repaint(CARD_HOVER)
        def on_leave(_): repaint(CARD_BG)
        def on_click(_): self._buka_detail(data)

        for widget in kandidat + [badge]:
            widget.bind("<Enter>", on_enter)
            widget.bind("<Leave>", on_leave)
            widget.bind("<Button-1>", on_click)
            try:
                widget.configure(cursor="hand2")
            except tk.TclError:
                pass

        return {"card": outer, "data": data}

    def _buka_detail(self, data):
        """Menampilkan jendela Popup (Toplevel) berisi gambar resolusi tinggi, data lengkap
        (termasuk harga weekday/weekend dan alamat), serta deskripsi utuh tanpa terpotong."""
        if self._popup is not None:
            try:
                if self._popup.winfo_exists():
                    self._popup.destroy()
            except Exception:
                pass

        popup = tk.Toplevel(self)
        self._popup = popup
        popup.title("Detail Wisata")
        popup.configure(bg=BG_COLOR)
        popup.resizable(False, False)
        
        w_width, w_height = 900, 520
        popup.geometry(f"{w_width}x{w_height}")
        popup.transient(self)
        popup.grab_set()

        popup.update_idletasks()
        x = (popup.winfo_screenwidth() // 2) - (w_width // 2)
        y = (popup.winfo_screenheight() // 2) - (w_height // 2)
        popup.geometry(f"{w_width}x{w_height}+{x}+{y}")

        popup.bind("<Escape>", lambda e: popup.destroy())

        shadow_frame = tk.Frame(popup, bg=SHADOW)
        shadow_frame.pack(fill="both", expand=True, padx=20, pady=20)
        
        main_frame = tk.Frame(shadow_frame, bg=CARD_BG)
        main_frame.pack(fill="both", expand=True, padx=(0, 2), pady=(0, 2))
        main_frame.grid_columnconfigure(0, weight=1, minsize=380)
        main_frame.grid_columnconfigure(1, weight=1)
        main_frame.grid_rowconfigure(0, weight=1)

        left_frame = tk.Frame(main_frame, bg=CARD_BG)
        left_frame.grid(row=0, column=0, sticky="nsew", padx=(24, 12), pady=24)

        thumb = load_thumb(data["nama"], ukuran=(360, 440))
        if thumb is not None:
            self._gambar_refs.append(thumb)
            lbl_img = tk.Label(left_frame, image=thumb, bg=CARD_BG, bd=0)
            lbl_img.image = thumb
            lbl_img.pack(fill="both", expand=True)
        else:
            ph = tk.Frame(left_frame, bg=IMAGE_BG)
            ph.pack(fill="both", expand=True)
            tk.Label(ph, text="\nGambar belum tersedia", bg=IMAGE_BG, fg=TEXT_MUTED, font=("Segoe UI", 12, "italic")).pack(expand=True)

        right_frame = tk.Frame(main_frame, bg=CARD_BG)
        right_frame.grid(row=0, column=1, sticky="nsew", padx=(12, 24), pady=24)

        top_title_frame = tk.Frame(right_frame, bg=CARD_BG)
        top_title_frame.pack(fill="x", pady=(0, 10))

        lbl_judul = tk.Label(top_title_frame, text=data["nama"], bg=CARD_BG, fg=PRIMARY, font=("Georgia", 20, "bold"), wraplength=340, justify="left", anchor="w")
        lbl_judul.pack(side="left", fill="x", expand=True)

        badge = tk.Label(top_title_frame, text=data["kategori"], bg=SECONDARY, fg=PRIMARY, font=("Segoe UI", 9, "bold"), padx=12, pady=4)
        badge.pack(side="right", anchor="ne")

        tk.Frame(right_frame, bg=SECONDARY, height=2).pack(fill="x", pady=(0, 16))

        details_frame = tk.Frame(right_frame, bg=CARD_BG)
        details_frame.pack(fill="x", pady=(0, 16))
        details_frame.grid_columnconfigure(2, weight=1)

        def add_detail(row, icon, label, value):
            tk.Label(details_frame, text=icon, bg=CARD_BG, fg=TEXT_MUTED, font=("Segoe UI", 10)).grid(row=row, column=0, sticky="nw", pady=4, padx=(0, 8))
            tk.Label(details_frame, text=label, bg=CARD_BG, fg=TEXT_MUTED, font=("Segoe UI", 10, "bold"), anchor="w").grid(row=row, column=1, sticky="nw", pady=4)
            tk.Label(details_frame, text=value, bg=CARD_BG, fg=TEXT_DARK, font=("Segoe UI", 10), anchor="w", wraplength=340, justify="left").grid(row=row, column=2, sticky="nw", pady=4, padx=(8, 0))

        add_detail(0, "⭐", "Rating", f"{data['rating']:.1f} / 5.0")
        add_detail(1, "🎟", "Tiket Weekday", data.get("harga_weekday", "-"))
        add_detail(2, "🎟", "Tiket Weekend", data.get("harga_weekend", "-"))
        add_detail(3, "📍", "Alamat", data.get("alamat", "-"))

        tk.Label(right_frame, text="Deskripsi Lengkap", bg=CARD_BG, fg=PRIMARY, font=("Georgia", 12, "bold"), anchor="w").pack(fill="x", pady=(0, 6))
        
        desc_frame = tk.Frame(right_frame, bg=BG_COLOR, highlightbackground=CARD_BORDER, highlightthickness=1)
        desc_frame.pack(fill="both", expand=True)
        
        txt = tk.Text(desc_frame, bg=BG_COLOR, fg=TEXT_DARK, font=("Segoe UI", 10), wrap="word", relief="flat", padx=12, pady=12, height=6)
        txt.insert("1.0", data["deskripsi"])
        txt.config(state="disabled")
        
        scroll = ttk.Scrollbar(desc_frame, orient="vertical", command=txt.yview)
        txt.configure(yscrollcommand=scroll.set)
        
        scroll.pack(side="right", fill="y")
        txt.pack(side="left", fill="both", expand=True)

    def _reset(self):
        """Mengembalikan kondisi aplikasi seperti awal: menghapus teks pencarian,
        mengatur filter kembali ke 'Semua Kategori', dan me-render ulang semua data."""
        self._placeholder_active = False
        self.search_var.set("")
        self.filter_var.set("Semua Kategori")
        self._entry.config(fg=TEXT_DARK)
        self._apply_filter(force=True)


# ==================== JALANKAN ====================
try:
    app.destroy()
except:
    pass

print(f"Dataset terbaca : {len(df)} baris × {len(df.columns)} kolom")
print(f"Kategori unik   : {KATEGORI_UNIK}")
print(f"Total destinasi : {len(DATA_WISATA)}")

app = AplikasiWisata()
app.mainloop()

Dataset terbaca : 71 baris × 8 kolom
Kategori unik   : ['Desa Wisata', 'Minat Khusus', 'Wisata Alam', 'Wisata Buatan', 'Wisata Budaya']
Total destinasi : 71
